# 00 — Construcción de occurrences.csv

Une los descargas GBIF de Aves y Mamíferos con las categorías IUCN obtenidas de Wikidata (notebook 01) y genera `data/processed/occurrences.csv`.

**Inputs:**
- `data/raw/gbif/0051889-260226173443078.csv` — Aves Bolivia (GBIF)
- `data/raw/gbif/mamiferos/0051890-260226173443078.csv` — Mamíferos Bolivia (GBIF)
- `data/raw/iucn/iucn_aves.csv` — Categorías IUCN aves (notebook 01)
- `data/raw/iucn/iucn_mamiferos.csv` — Categorías IUCN mamíferos (notebook 01)

**Output:**
- `data/processed/occurrences.csv`

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path('../data')
OUT  = DATA / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

# Columnas a conservar del dump GBIF
COLS = [
    'gbifID', 'species', 'taxonRank', 'class',
    'decimalLatitude', 'decimalLongitude',
    'coordinateUncertaintyInMeters',
    'countryCode', 'stateProvince',
    'year', 'month', 'day',
    'basisOfRecord', 'establishmentMeans',
]

In [ ]:
# ── Cargar GBIF aves ──────────────────────────────────────────────
print('Cargando aves GBIF...')
aves_path = DATA / 'raw/gbif/0051889-260226173443078.csv'
df_aves = pd.read_csv(aves_path, sep='\t', low_memory=False, usecols=COLS)
df_aves['class'] = 'Aves'  # asegurar etiqueta uniforme
print(f'  Registros crudos: {len(df_aves):,}')

# Solo registros a nivel especie con coordenadas
df_aves = df_aves[
    (df_aves['taxonRank'] == 'SPECIES') &
    df_aves['species'].notna() &
    df_aves['decimalLatitude'].notna() &
    df_aves['decimalLongitude'].notna()
]
print(f'  Tras filtro rank/coords: {len(df_aves):,} | especies: {df_aves["species"].nunique():,}')

In [ ]:
# ── Cargar GBIF mamíferos ─────────────────────────────────────────
print('Cargando mamiferos GBIF...')
mam_path = DATA / 'raw/gbif/mamiferos/0051890-260226173443078.csv'
df_mam = pd.read_csv(mam_path, sep='\t', low_memory=False, usecols=COLS)
df_mam['class'] = 'Mammalia'  # asegurar etiqueta uniforme
print(f'  Registros crudos: {len(df_mam):,}')

df_mam = df_mam[
    (df_mam['taxonRank'] == 'SPECIES') &
    df_mam['species'].notna() &
    df_mam['decimalLatitude'].notna() &
    df_mam['decimalLongitude'].notna()
]
print(f'  Tras filtro rank/coords: {len(df_mam):,} | especies: {df_mam["species"].nunique():,}')

In [ ]:
# ── Unir con categorías IUCN ──────────────────────────────────────
iucn_aves = pd.read_csv(DATA / 'raw/iucn/iucn_aves.csv')
iucn_mam  = pd.read_csv(DATA / 'raw/iucn/iucn_mamiferos.csv')

print(f'IUCN aves:      {len(iucn_aves):,} especies, {iucn_aves["iucn_categoria"].notna().sum()} con categoria')
print(f'IUCN mamiferos: {len(iucn_mam):,}  especies, {iucn_mam["iucn_categoria"].notna().sum()} con categoria')

df_aves = df_aves.merge(iucn_aves[['species', 'iucn_categoria']], on='species', how='left')
df_mam  = df_mam.merge(iucn_mam[['species',  'iucn_categoria']], on='species', how='left')

In [ ]:
# ── Concatenar y guardar ─────────────────────────────────────────
occ = pd.concat([df_aves, df_mam], ignore_index=True)

print(f'Total ocurrencias: {len(occ):,}')
print(f'Especies:          {occ["species"].nunique():,}')
print(f'  Aves:      {occ[occ["class"]=="Aves"]["species"].nunique():,}')
print(f'  Mammalia:  {occ[occ["class"]=="Mammalia"]["species"].nunique():,}')
print(f'\nDistribucion IUCN:')
print(occ['iucn_categoria'].value_counts(dropna=False).to_string())

out_path = OUT / 'occurrences.csv'
occ.to_csv(out_path, index=False)
print(f'\nGuardado: {out_path}  ({out_path.stat().st_size / 1024 / 1024:.1f} MB)')